# ============================================================
# dl-24f3004686-notebook-t22026.ipynb
# ============================================================

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


In [2]:
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [3]:
"""
TF-IDF + Logistic Regression Model
"""

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


class TFIDFModel:

    def __init__(self):

        self.vectorizer = TfidfVectorizer(
            max_features=60000,
            ngram_range=(1,3),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )

        self.model = LogisticRegression(
            C=2.5,
            solver="liblinear",
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        )
        
    def prepare_training_data(self, train_df):

        X_train = []
        y_train = []

        for _, row in train_df.iterrows():

            prompt = row["prompt"]

            for option in ["A", "B", "C", "D", "E"]:

                X_train.append(
                    prompt + " " + str(row[option])
                )

                y_train.append(
                    1 if row["answer"] == option else 0
                )

        return X_train, y_train
    
    def train(self, train_df):

        X_train, y_train = self.prepare_training_data(train_df)

        X_train_vec = self.vectorizer.fit_transform(X_train)

        self.model.fit(X_train_vec, y_train)

        print("Training Complete!")
        
        
    def predict_top3(self, test_df):

        predictions = []

        for _, row in test_df.iterrows():

            scores = {}

            for option in ["A", "B", "C", "D", "E"]:

                text = row["prompt"] + " " + str(row[option])

                vec = self.vectorizer.transform([text])

                score = self.model.predict_proba(vec)[0][1]

                scores[option] = score

            ranked = sorted(
                scores,
                key=scores.get,
                reverse=True
            )

            predictions.append(ranked[:3])

        return predictions
    

In [4]:
model = TFIDFModel()

model.train(train)

Training Complete!


In [5]:
predictions = model.predict_top3(test)

predictions[:5]

[['A', 'E', 'B'],
 ['B', 'E', 'A'],
 ['B', 'E', 'D'],
 ['E', 'C', 'D'],
 ['C', 'A', 'D']]

In [6]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(p) for p in predictions]
})

submission.head()

,ID,Prediction
0,1,A E B
1,2,B E A
2,3,B E D
3,4,E C D
4,5,C A D


In [7]:
submission.to_csv("submission.csv", index=False)

# ============================================================
# dl-24f3004686-notebook-t22026 neural.ipynb
# ============================================================

In [1]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [2]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [4]:
train_part, valid_part = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train["answer"]
)

print(train_part.shape)
print(valid_part.shape)

(1600, 8)
(400, 8)


In [5]:
X_train = []
y_train = []

for _, row in train_part.iterrows():

    for option in ["A","B","C","D","E"]:

        text = row["prompt"] + " " + str(row[option])

        X_train.append(text)

        y_train.append(
            1 if row["answer"] == option else 0
        )

print(len(X_train))
print(len(y_train))

8000
8000


In [6]:
vectorizer = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(X_train)

print(X_train_vec.shape)

(8000, 28429)


In [7]:
X_train_tensor = torch.FloatTensor(
    X_train_vec.toarray()
)

y_train_tensor = torch.LongTensor(y_train)

print(X_train_tensor.shape)
print(y_train_tensor.shape)


torch.Size([8000, 28429])
torch.Size([8000])


In [8]:
class MCQNet(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size, 512),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(512,128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128,2)
        )
    def forward(self,x):

        return self.network(x)

In [9]:
input_size = X_train_tensor.shape[1]

model = MCQNet(input_size)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)

MCQNet(
  (network): Sequential(
    (0): Linear(in_features=28429, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [10]:
epochs = 5

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    outputs = model(X_train_tensor)

    loss = criterion(
        outputs,
        y_train_tensor
    )

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1}/{epochs}  Loss: {loss.item():.4f}"
    )

Epoch 1/5  Loss: 0.7056
Epoch 2/5  Loss: 0.6927
Epoch 3/5  Loss: 0.6792
Epoch 4/5  Loss: 0.6635
Epoch 5/5  Loss: 0.6453


In [11]:
X_valid = []
y_valid = []

for _, row in valid_part.iterrows():

    for option in ["A","B","C","D","E"]:

        X_valid.append(
            row["prompt"] + " " + str(row[option])
        )

        y_valid.append(
            1 if row["answer"] == option else 0
        )

print(len(X_valid))

2000


In [12]:
X_valid_vec = vectorizer.transform(X_valid)

X_valid_tensor = torch.FloatTensor(
    X_valid_vec.toarray()
)

print(X_valid_tensor.shape)

torch.Size([2000, 28429])


In [13]:
model.eval()

with torch.no_grad():

    outputs = model(X_valid_tensor)

    probs = torch.softmax(outputs, dim=1)[:,1].numpy()

print(probs[:10])

[0.46151394 0.46077073 0.47110873 0.46123117 0.46063083 0.45113865
 0.4442799  0.44893557 0.48735857 0.45063907]


In [14]:
predictions = []

idx = 0

for _, row in valid_part.iterrows():

    scores = {}

    for option in ["A", "B", "C", "D", "E"]:

        scores[option] = probs[idx]

        idx += 1

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    predictions.append(ranked[:3])

print(predictions[:5])

[['C', 'A', 'D'], ['D', 'A', 'E'], ['E', 'C', 'A'], ['D', 'E', 'A'], ['B', 'D', 'C']]


In [15]:
def map3(actual, predicted):

    score = 0.0

    for truth, preds in zip(actual, predicted):

        for rank, pred in enumerate(preds[:3]):

            if pred == truth:

                score += 1 / (rank + 1)

                break

    return score / len(actual)

In [16]:
actual = valid_part["answer"].tolist()

score = map3(actual, predictions)

print(f"Validation MAP@3: {score:.5f}")

Validation MAP@3: 0.93875


In [17]:
X_full = []
y_full = []

for _, row in train.iterrows():

    for option in ["A", "B", "C", "D", "E"]:

        X_full.append(
            row["prompt"] + " " + str(row[option])
        )

        y_full.append(
            1 if row["answer"] == option else 0
        )

In [18]:
vectorizer = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1,5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_full_vec = vectorizer.fit_transform(X_full)

X_full_tensor = torch.FloatTensor(
    X_full_vec.toarray()
)

y_full_tensor = torch.LongTensor(y_full)

model = MCQNet(X_full_tensor.shape[1])

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [19]:
epochs = 5

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    outputs = model(X_full_tensor)

    loss = criterion(outputs, y_full_tensor)

    loss.backward()

    optimizer.step()

    print(f"Epoch {epoch+1}: {loss.item():.4f}")


Epoch 1: 0.6570
Epoch 2: 0.6421
Epoch 3: 0.6239
Epoch 4: 0.6035
Epoch 5: 0.5793


In [20]:
X_test = []

for _, row in test.iterrows():

    for option in ["A","B","C","D","E"]:

        X_test.append(
            row["prompt"] + " " + str(row[option])
        )

In [21]:
X_test_vec = vectorizer.transform(X_test)

X_test_tensor = torch.FloatTensor(
    X_test_vec.toarray()
)

In [22]:
model.eval()

with torch.no_grad():

    outputs = model(X_test_tensor)

    probs = torch.softmax(outputs, dim=1)[:,1].numpy()

In [23]:
predictions = []

idx = 0

for _, row in test.iterrows():

    scores = {}

    for option in ["A","B","C","D","E"]:

        scores[option] = probs[idx]
        idx += 1

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    predictions.append(ranked[:3])

In [24]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(p) for p in predictions]
})

submission.head()


,ID,Prediction
0,1,A E B
1,2,B E A
2,3,B E A
3,4,E C D
4,5,C A B


In [25]:
submission.to_csv(
    "submission_neural.csv",
    index=False
)

print("Submission saved!")

Submission saved!


In [26]:
import os

print(os.listdir("/kaggle/working"))

['__notebook__.ipynb', 'submission_neural.csv']


In [27]:
submission.to_csv("/kaggle/working/submission.csv", index=False)

import os
print(os.path.exists("/kaggle/working/submission.csv"))

True


# ============================================================
# dl-24f3004686-notebook-t22026 pre.ipynb
# ============================================================

In [1]:
!pip -q install transformers datasets accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [2]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(train_df.shape)
print(test_df.shape)
train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
train_df["text"] = (
    train_df["prompt"].fillna("") +
    " [SEP] A: " + train_df["A"].fillna("") +
    " [SEP] B: " + train_df["B"].fillna("") +
    " [SEP] C: " + train_df["C"].fillna("") +
    " [SEP] D: " + train_df["D"].fillna("") +
    " [SEP] E: " + train_df["E"].fillna("")
)

test_df["text"] = (
    test_df["prompt"].fillna("") +
    " [SEP] A: " + test_df["A"].fillna("") +
    " [SEP] B: " + test_df["B"].fillna("") +
    " [SEP] C: " + test_df["C"].fillna("") +
    " [SEP] D: " + test_df["D"].fillna("") +
    " [SEP] E: " + test_df["E"].fillna("")
)

train_df[["text", "answer"]].head()

,text,answer
0,Pick the best possible answer: What is Martin ...,B
1,What is accelerator-based light-ion fusion? [S...,A
2,Determine the correct option: What is the term...,C
3,Select the most accurate option: What is Marti...,B
4,Identify the correct statement: What is the co...,A


In [5]:
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train_df["label"] = train_df["answer"].map(label_map)

train_df[["answer", "label"]].head()

,answer,label
0,B,1
1,A,0
2,C,2
3,B,1
4,A,0


In [6]:
from sklearn.model_selection import train_test_split

train_data, valid_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"]
)

print(train_data.shape)
print(valid_data.shape)

(1600, 10)
(400, 10)


In [7]:
MODEL_NAME = "FacebookAI/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
train_dataset = Dataset.from_pandas(
    train_data[["text", "label"]]
)

valid_dataset = Dataset.from_pandas(
    valid_data[["text", "label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]]
)

In [9]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [10]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)

print("Model loaded successfully!")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="no",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=50,

    report_to="none",

    fp16=torch.cuda.is_available()
)

In [13]:
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)

    acc = (preds == labels).mean()

    return {"accuracy": acc}

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

In [15]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,2.970431,2.388458,0.540000
2,0.644145,0.182468,1.000000
3,0.062880,0.032042,1.000000


TrainOutput(global_step=300, training_loss=1.4823667152722677, metrics={'train_runtime': 156.7677, 'train_samples_per_second': 30.619, 'train_steps_per_second': 1.914, 'total_flos': 631483541913600.0, 'train_loss': 1.4823667152722677, 'epoch': 3.0})

In [16]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.03204205632209778, 'eval_accuracy': 1.0, 'eval_runtime': 4.1635, 'eval_samples_per_second': 96.072, 'eval_steps_per_second': 6.004, 'epoch': 3.0}


In [17]:
# Predict on test set
test_predictions = trainer.predict(test_dataset)

# Extract logits
test_logits = test_predictions.predictions

print("Test logits shape:", test_logits.shape)

Test logits shape: (500, 5)


In [18]:
import numpy as np

# Convert logits to probabilities
probabilities = torch.softmax(
    torch.tensor(test_logits),
    dim=1
).numpy()

# Label mapping
labels = np.array(["A", "B", "C", "D", "E"])

# Top-3 predictions
top3 = np.argsort(probabilities, axis=1)[:, -3:][:, ::-1]

prediction_strings = [
    " ".join(labels[idx])
    for idx in top3
]

print(prediction_strings[:5])

['A D C', 'B D A', 'B C D', 'E C D', 'C D A']


In [19]:
submission = sample_submission.copy()

submission["Prediction"] = prediction_strings

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,Prediction
0,1,A D C
1,2,B D A
2,3,B C D
3,4,E C D
4,5,C D A


In [20]:
import os

print(submission.shape)
print(sample_submission.shape)

print(submission.columns)

print(os.listdir("/kaggle/working"))

(500, 2)
(500, 2)
Index(['ID', 'Prediction'], dtype='object')
['submission.csv', 'results', '__notebook__.ipynb']
